# 2021 Chignik earthquake — Gulf of Alaska DART arrival times

This notebook computes the first-arrival travel time map for the **2021 Chignik
M8.2 earthquake** (29 July 2021, epicenter 55.36°N 157.89°W) across the Gulf
of Alaska, then reads off the predicted tsunami arrival times at three nearby
DART pressure-sensor buoys operated by NOAA/NDBC.

| DART | Lat | Lon |
|------|-----|-----|
| 46414 | 53.68°N | 152.35°W |
| 46409 | 55.34°N | 148.57°W |
| 46410 | 57.65°N | 143.69°W |

**Bathymetry:** ETOPO2 4 arc-minute grid (`data/NE_pacific_4arcmin.nc`).

> Requires `scipy` and `netCDF4`:  `pip install -e ".[examples]"`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import TsunamiTrace as tt
%matplotlib inline

## Load bathymetry

The NetCDF file stores longitudes in the 0–360° convention (180–260°E).
We shift them to the –180/180°W convention for both plotting and ray tracing.

In [ ]:
print('Loading data/NE_pacific_4arcmin.nc ...', end='', flush=True)
lon_arr, lat_arr, depth_tracing = tt.load_bathymetry('../data/NE_pacific_4arcmin.nc')
print(' done.')

# Shift 180–260°E → –180 to –100°W
lon_arr = lon_arr - 360.0

# depth_grid: (n_lat, n_lon) with geographic sign (negative = ocean) for plotting
depth_grid = -depth_tracing.T

n_lon, n_lat = depth_tracing.shape
print(f'Grid : {n_lon} × {n_lat}  (spacing {lon_arr[1]-lon_arr[0]:.4f}°)')
print(f'Lon  : {lon_arr[0]:.2f} to {lon_arr[-1]:.2f} °')
print(f'Lat  : {lat_arr[0]:.2f} to {lat_arr[-1]:.2f} °N')

## Source and DART locations

In [ ]:
# 2021 Chignik earthquake (M8.2, 29 July 2021) epicenter
SOURCE_LON = -157.89
SOURCE_LAT =  55.36

# DART buoy positions (NDBC-confirmed)
DARTS = {
    '46414': {'lon': -152.35, 'lat': 53.68},
    '46409': {'lon': -148.57, 'lat': 55.34},
    '46410': {'lon': -143.69, 'lat': 57.65},
}

# Map extent — Gulf of Alaska, regional focus
LON_MIN, LON_MAX = -165.0, -138.0
LAT_MIN, LAT_MAX =   49.0,   63.0

print(f'Source  : {SOURCE_LON}°, {SOURCE_LAT}°N')
for name, d in DARTS.items():
    print(f'DART {name}: {d["lat"]:.2f}°N, {abs(d["lon"]):.2f}°W')

## Overview map — bathymetry, epicenter, and DART locations

In [ ]:
vmin_d = float(np.nanmin(depth_grid))
vmax_d = float(np.nanmax(depth_grid))
norm_bathy = mcolors.TwoSlopeNorm(vmin=vmin_d, vcenter=0, vmax=max(vmax_d, 1))

fig, ax = plt.subplots(figsize=(10, 6))

ax.contourf(lon_arr, lat_arr, depth_grid,
            levels=100, cmap='gist_earth', norm=norm_bathy)
ax.contour(lon_arr, lat_arr, depth_grid,
           levels=[0], colors='k', linewidths=0.8)

# Epicenter
ax.plot(SOURCE_LON, SOURCE_LAT,
        marker=(5, 1), markersize=14,
        markerfacecolor='red', markeredgecolor='k', markeredgewidth=0.8,
        linestyle='none', zorder=6, label='Chignik M8.2 epicenter')

# DART buoys
for name, d in DARTS.items():
    ax.plot(d['lon'], d['lat'],
            marker='^', markersize=11,
            markerfacecolor='cyan', markeredgecolor='k', markeredgewidth=0.9,
            linestyle='none', zorder=6)
    ax.text(d['lon'] + 0.4, d['lat'] + 0.4, f'DART {name}',
            fontsize=8, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      alpha=0.8, edgecolor='0.5'))

ax.set_xlim([LON_MIN, LON_MAX])
ax.set_ylim([LAT_MIN, LAT_MAX])
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Gulf of Alaska — 2021 Chignik M8.2 epicenter and DART buoys')
ax.set_aspect('equal')
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

## Tsunami ray tracing

We fan 360 rays at 1° azimuth spacing from the Chignik epicenter and integrate
for 3 hours — long enough for the wave to reach all three DARTs with time to
spare, but short enough to keep the run fast and the map regional.

In [ ]:
DT       = 60.0                           # time step (s)
MAX_TIME = 3 * 3600.0                     # 3 hours
AZIMUTHS = np.arange(0, 360, 1.0, dtype=float)   # 360 rays

print(f'Source   : {SOURCE_LON}°, {SOURCE_LAT}°N')
print(f'Rays     : {len(AZIMUTHS)}  (every {AZIMUTHS[1]-AZIMUTHS[0]:.1f}°)')
print(f'dt       : {DT:.0f} s    max time : {MAX_TIME/3600:.0f} h')
print('Tracing ...', end='', flush=True)

ray_lon, ray_lat, _ = tt.trace_rays(
    lon_arr, lat_arr, depth_tracing,
    DT, MAX_TIME,
    SOURCE_LON, SOURCE_LAT,
    AZIMUTHS,
)

n_done = int(np.sum(~np.isnan(ray_lon[:, -1])))
print(f'  done.  {n_done}/{len(AZIMUTHS)} rays reached max_time.')

## Grid travel times

In [ ]:
BIN_DEG = 0.15   # output cell size in degrees (~17 km)

print('Gridding travel times ...', end='', flush=True)
lon_bin, lat_bin, travel_time = tt.grid_travel_times(
    ray_lon, ray_lat, DT,
    lon_arr, lat_arr, depth_tracing,
    bin_deg=BIN_DEG,
    fill=True,
)
print(f' done.  Output grid: {len(lon_bin)} × {len(lat_bin)}')

## Predicted arrival times at each DART

We find the travel-time grid bin nearest to each buoy and read off the
first-arrival time.

In [ ]:
print(f'{"DART":<8}  {"Position":^22}  {"Arrival time":>14}  {"(min)":>7}')
print('-' * 58)

for name, dart in DARTS.items():
    i_lat = int(np.argmin(np.abs(lat_bin - dart['lat'])))
    i_lon = int(np.argmin(np.abs(lon_bin - dart['lon'])))
    t_hr  = float(travel_time[i_lat, i_lon])
    t_min = t_hr * 60.0
    hrs   = int(t_min // 60)
    mins  = int(round(t_min % 60))

    dart['arrival_hr']    = t_hr
    dart['arrival_label'] = f'{hrs}h {mins:02d}min' if hrs > 0 else f'{mins} min'

    pos = f'{dart["lat"]:.2f}°N  {abs(dart["lon"]):.2f}°W'
    print(f'DART {name}  {pos:^22}  {dart["arrival_label"]:>14}  ({t_min:>5.1f} min)')

## Travel time map with DART arrival times

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

# ── bathymetry background ─────────────────────────────────────────────────────
ax.contourf(lon_arr, lat_arr, depth_grid,
            levels=100, cmap='gist_earth', norm=norm_bathy, alpha=0.4)
ax.contour(lon_arr, lat_arr, depth_grid,
           levels=[0], colors='k', linewidths=0.8)

# ── travel time filled contours ───────────────────────────────────────────────
tt_max   = MAX_TIME / 3600.0
cf = ax.contourf(lon_bin, lat_bin, travel_time,
                 levels=np.arange(0, tt_max + 0.25, 0.25),
                 cmap='plasma_r', alpha=0.78, extend='max')

# Isochrone lines every 30 min
cs = ax.contour(lon_bin, lat_bin, travel_time,
                levels=np.arange(0.5, tt_max + 0.5, 0.5),
                colors='white', linewidths=0.8, alpha=0.9)
ax.clabel(cs, fmt='%.1f h', fontsize=7, inline=True)

# ── epicenter ─────────────────────────────────────────────────────────────────
ax.plot(SOURCE_LON, SOURCE_LAT,
        marker=(5, 1), markersize=14,
        markerfacecolor='red', markeredgecolor='k', markeredgewidth=0.8,
        linestyle='none', zorder=7, label='Chignik M8.2 epicenter')

# ── DART buoys with arrival time callouts ─────────────────────────────────────
# Text offsets (degrees) chosen to keep labels readable and non-overlapping
text_offsets = {
    '46414': (+1.2, -1.8),   # south-east of marker
    '46409': (+1.2, +1.2),   # north-east
    '46410': (+1.2, +1.0),   # north-east
}

for name, dart in DARTS.items():
    ax.plot(dart['lon'], dart['lat'],
            marker='^', markersize=11,
            markerfacecolor='cyan', markeredgecolor='k', markeredgewidth=0.9,
            linestyle='none', zorder=7)
    dlon, dlat = text_offsets[name]
    ax.annotate(
        f'DART {name}\n{dart["arrival_label"]}',
        xy=(dart['lon'], dart['lat']),
        xytext=(dart['lon'] + dlon, dart['lat'] + dlat),
        fontsize=9, fontweight='bold', color='k',
        ha='left', va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  alpha=0.88, edgecolor='0.4'),
        arrowprops=dict(arrowstyle='->', color='k', lw=0.8),
        zorder=8,
    )

ax.set_xlim([LON_MIN, LON_MAX])
ax.set_ylim([LAT_MIN, LAT_MAX])
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(
    f'2021 Chignik M8.2 — first-arrival travel times\n'
    f'Source: {abs(SOURCE_LON):.2f}°W, {SOURCE_LAT:.2f}°N  ·  '
    f'{len(AZIMUTHS)} rays  ·  dt={DT:.0f} s  ·  bin={BIN_DEG}°'
)
ax.set_aspect('equal')
ax.legend(fontsize=9, loc='lower left')

cbar = fig.colorbar(cf, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Travel time (hours)')
plt.tight_layout()
plt.show()